# Topic: SQL Injection Prevention - Blind vs Union SQLi mechanics, input constraints, and database boundary parameterizations
 
## 1. UNION VS BLIND SQL INJECTION
- **Union-Based SQLi**: The attacker appends `UNION SELECT` statement blocks to merge their query outputs into the application's visual responses, extracting columns from other tables directly.
- **Blind SQLi**:
  - Occurs when the web application does not output query database values directly in the responses.
  - **Boolean-Based Blind SQLi**: Attackers ask True/False questions (e.g. `' AND (SELECT SUBSTR(password, 1, 1) FROM users WHERE username='admin') = 'a'`). If the page loads normally, the guess was correct; if it fails, it was incorrect.
  - **Time-Based Blind SQLi**: Attackers inject sleep commands (e.g. `sqlite3` `sleep` extensions or PostgreSQL `pg_sleep(5)`). If the page response delays by 5 seconds, the condition evaluates to True.
 
## 2. COMPREHENSIVE MITIGATION
- **Prepared Statements**: Forces database parsers to treat parameters as raw data.
- **Input Constraints**: Reject strings that do not conform to expected shapes.
 
---


In [ ]:
import sqlite3
import time

# 1. Setup temporary database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, username TEXT, balance REAL)")
cursor.execute("INSERT INTO accounts (username, balance) VALUES ('admin', 9999.0)")
cursor.execute("INSERT INTO accounts (username, balance) VALUES ('user1', 120.0)")
conn.commit()



In [ ]:
# 2. Vulnerable User Verification (Boolean-Based Blind SQLi candidate)
def check_user_exists(username):
    """Vulnerable user checking concatenating input string."""
    query = f"SELECT * FROM accounts WHERE username = '{username}'"
    
    cursor.execute(query)
    result = cursor.fetchone()
    # Only returns True (exists) or False (not exists)
    # The attacker doesn't see database outputs, but can run Boolean Blind queries!
    return result is not None

print("--- Testing Boolean Blind SQLi ---")
# Normal check
print(f"Admin exists? {check_user_exists('admin')}")

# Attacker verifies if first char of admin's balance is 9:
# Input: admin' AND SUBSTR(CAST(balance AS TEXT), 1, 1) = '9
blind_payload = "admin' AND SUBSTR(CAST(balance AS TEXT), 1, 1) = '9"
print(f"Blind query evaluation (Is first char 9?): {check_user_exists(blind_payload)}")
# Expected: True! Attacker now knows the balance starts with 9 without viewing the row.



In [ ]:
# 3. Secure User Verification (Parameterized)
def secure_check_user_exists(username):
    """Secure user checking using parameterized parameter bindings."""
    query = "SELECT * FROM accounts WHERE username = ?"
    cursor.execute(query, (username,))
    result = cursor.fetchone()
    return result is not None

print("\n--- Testing Secure Parameterized Verification ---")
print(f"Blind query evaluation (Is first char 9?): {secure_check_user_exists(blind_payload)}")
# Expected: False! Parameterization treated the entire malicious string as a single username literal.

conn.close()
